In [653]:
import pandas as pd

df = pd.read_csv('Book1.csv')
df.head()

,date,product,quantity_sold,price
0,1/1/2024,A,10,100
1,1/2/2024,A,12,100
2,1/3/2024,B,5,200
3,1/4/2024,A,15,100
4,1/5/2024,B,7,200


In [654]:
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   date           10 non-null     str  
 1   product        10 non-null     str  
 2   quantity_sold  10 non-null     int64
 3   price          10 non-null     int64
dtypes: int64(2), str(2)
memory usage: 452.0 bytes


,quantity_sold,price
count,10.000000,10.000000
mean,12.300000,140.000000
std,6.129165,51.639778
min,5.000000,100.000000
25%,7.250000,100.000000
50%,11.000000,100.000000
75%,17.250000,200.000000
max,22.000000,200.000000


Convert Date (Now Python understands time)

In [655]:
df['date'] = pd.to_datetime(df['date'])
print(df['date'])

0   2024-01-01
1   2024-01-02
2   2024-01-03
3   2024-01-04
4   2024-01-05
5   2024-01-06
6   2024-01-07
7   2024-01-08
8   2024-01-09
9   2024-01-10
Name: date, dtype: datetime64[us]


Create New Features 
Now model can learn patterns like:
1.weekends
2.monthly trends

In [656]:
df["day"] = df["date"].dt.day
df["month"] = df["date"].dt.month
df['day_of_week'] = df['date'].dt.dayofweek

Basic Analysis :
Total sales per product

In [657]:
print(df.groupby('product')['quantity_sold'].sum())

product
A    97
B    26
Name: quantity_sold, dtype: int64


Average sales

In [658]:
print(df['quantity_sold'].mean())

12.3


Highest sales day

In [659]:
print(df[df['quantity_sold'] == df['quantity_sold'].max()])

        date product  quantity_sold  price  day  month  day_of_week
9 2024-01-10       A             22    100   10      1            2


In [660]:
df =pd.get_dummies(df, columns=['product'])

In [661]:
X = df[["price", "day", "month", "day_of_week", "product_A", "product_B"]]
Y = df["quantity_sold"]

In [662]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

In [663]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, Y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [664]:
predictions = model.predict(X_test)
print("Actual:", list(Y_test)[:5])
print('predicted:', predictions)

Actual: [6, 12]
predicted: [ 9.60784314 12.42647059]


In [665]:
print('Accuracy:', model.score(X_test, Y_test))

Accuracy: 0.26675504079627566


In [666]:
new_data = [[100,11,1,3,1,0]]
print(model.predict(new_data))
print('Accuracy:', model.score(X_test, Y_test))

[23.08823529]
Accuracy: 0.26675504079627566


C:\Users\ASUS\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [667]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [668]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

In [669]:
from sklearn.metrics import mean_absolute_error

Y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(Y_test, Y_pred))

MAE: 2.0171568627450984


In [670]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor()
model.fit(X_train, Y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [671]:
print('Accuracy : ' , model.score(X_test,Y_test))

Accuracy :  -1.1288722222222218
